# 01 - Data preparation

In [1]:
import pandas as pd
import numpy as np

# === CONFIG ===
INPUT_CSV = 'AB_NYC_2019.csv'
OUTPUT_CSV = 'AB_NYC_2019_cleaned.csv'

ROOM_TYPE_MAP = {'Entire home/apt': 1, 'Private room': 2, 'Shared room': 3}
NEIGHBOURHOOD_GROUP_MAP = {'Manhattan': 1, 'Brooklyn': 2, 'Queens': 3, 'Bronx': 4, 'Staten Island': 5}

MIN_NIGHTS_QUANTILE = 0.99

MODELLING_COLS = [
    'log_price', 'room_type_code', 'neighbourhood_group_code',
    'latitude', 'longitude', 'minimum_nights',
    'number_of_reviews', 'reviews_per_month',
    'availability_365', 'calculated_host_listings_count'
]
# === END CONFIG ===

In [2]:
def log_table(title, df, decimals=None, index=True):
    print()
    print('--- ' + title + ' ---')
    if decimals is not None:
        print(df.round(decimals).to_string(index=index))
    else:
        print(df.to_string(index=index))

In [3]:
df = pd.read_csv(INPUT_CSV)

In [4]:
# Task 2.1 - reviews_per_month = 0 where no reviews; last_review kept as NaN (date field, not in modelling cols)
df.loc[df['number_of_reviews'] == 0, 'reviews_per_month'] = 0

# Task 2.2 - drop rows with NaN in name or host_name
df = df[df['name'].notna() & df['host_name'].notna()]

# Task 2.3 - drop price = 0
df = df[df['price'] > 0]

# Task 2.4 - log target
df['log_price'] = np.log1p(df['price'])

# Task 2.5 - numeric encoding for categoricals
df['room_type_code'] = df['room_type'].map(ROOM_TYPE_MAP)
df['neighbourhood_group_code'] = df['neighbourhood_group'].map(NEIGHBOURHOOD_GROUP_MAP)

# Task 2.6 - q99 cap on minimum_nights
q99 = df['minimum_nights'].quantile(MIN_NIGHTS_QUANTILE)
df['minimum_nights'] = df['minimum_nights'].clip(upper=q99)

In [5]:
# Task 2.7 - save cleaned CSV
df.to_csv(OUTPUT_CSV, index=False)

In [6]:
# Task 2.8 - summary
print('Final row count:', len(df))
print('Final columns:', list(df.columns))
print()
print('NaN per modelling column:')
for col in MODELLING_COLS:
    print(f'  {col}: {df[col].isna().sum()}')
print()
print(f'room_type_map: {ROOM_TYPE_MAP}')
print(f'neighbourhood_group_map: {NEIGHBOURHOOD_GROUP_MAP}')
print(f'q99 threshold for minimum_nights: {q99}')
log_table('Describe on modelling columns', df[MODELLING_COLS].describe())

Final row count: 48847
Final columns: ['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'log_price', 'room_type_code', 'neighbourhood_group_code']

NaN per modelling column:
  log_price: 0
  room_type_code: 0
  neighbourhood_group_code: 0
  latitude: 0
  longitude: 0
  minimum_nights: 0
  number_of_reviews: 0
  reviews_per_month: 0
  availability_365: 0
  calculated_host_listings_count: 0

room_type_map: {'Entire home/apt': 1, 'Private room': 2, 'Shared room': 3}
neighbourhood_group_map: {'Manhattan': 1, 'Brooklyn': 2, 'Queens': 3, 'Bronx': 4, 'Staten Island': 5}
q99 threshold for minimum_nights: 45.0

--- Describe on modelling columns ---
          log_price  room_type_code  neighbourhood_group_code      latitude     longitude  minimum_nights  number_of_reviews  reviews_per_month  av